# Adidas Global Retail — Competitive Analysis

## Objective

We are simulating a real-world scenario in which a **sportswear company wants to understand 
how to position itself with respect to Adidas before entering the market.** 
>  **This is a personal project.**

## Dataset 

The dataset `Adidas_Global.csv` contains product and pricing information for Adidas items 
across multiple countries and currencies. It consists of **44,888 records** with **20 features** 
including product details, regional pricing, availability, and discount data.

The CSV is loaded into a **MySQL database** to enable structured querying 
and analysis rather than working directly on the raw file.

> ⚠️ **Note:** A `DtypeWarning` was raised during import, indicating mixed data types in some 
> columns. This will be addressed in the data cleaning phase.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("C:/Users/lorem/Desktop/adidas-retail-analysis/data/Adidas_Global.csv")

engine = create_engine("mysql+pymysql://root:@localhost:3306/adidas_retail")

df.to_sql("adidas_global", con=engine, if_exists="replace", index=False)

C:\Users\lorem\AppData\Local\Temp\ipykernel_16408\2544219958.py:4: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/lorem/Desktop/adidas-retail-analysis/data/Adidas_Global.csv")


44888

## Data Cleaning — SQL Query

Before analysis, the raw dataset was cleaned and transformed using SQL.

**Key operations:**

**1. Deduplication** — `MIN(price_local)` and `GROUP BY product_id` 
keep only the lowest price per unique product.

**2. Currency conversion to EUR** — A `CASE` statement converts all 
prices to EUR using fixed exchange rates.

**3. Discount handling** — `COALESCE(MAX(discount_pct), 0)` replaces 
NULL discounts with 0.

**4. Filtering** — Only products with a valid `subcategory` and 
relevant sport categories are included.

```sql
SELECT 
    product_id,
    product_name,
    category,
    subcategory,
    gender_segment,
    product_url,
    currency,
    MIN(price_local),
    ROUND(MIN(price_local) * CASE currency
      WHEN 'EUR' THEN 1.0
      WHEN 'USD' THEN 0.92
      WHEN 'GBP' THEN 1.17
      WHEN 'BRL' THEN 0.17
      WHEN 'CNY' THEN 0.13
      WHEN 'INR' THEN 0.011
      WHEN 'JPY' THEN 0.0062
      WHEN 'KRW' THEN 0.00068
      WHEN 'MXN' THEN 0.046
      ELSE 1.0
    END, 2) as price_eur,
    REPLACE(CAST(COALESCE(MAX(discount_pct), 0) AS CHAR), '.', ',') as discount_pct
FROM adidas_global
WHERE subcategory IS NOT NULL 
  AND category IN ('Running','Basketball','Golf','Fitness & Training', 
                   'Football', 'Motorsport', 'Tennis')
GROUP BY 
    product_id,
    category,
    subcategory,
    gender_segment,
    currency
```

The query output was exported as a CSV file named `query_sport_category_eur_converted_and_clean_data.csv` 
and loaded into Tableau as a new data source for further visual analysis.

The two charts below show the average price and total product count 
for each Adidas sport category.

**What we observe:**
- **Running** dominates the catalog by volume, it is by far the largest 
category in terms of number of products.
- Average prices are relatively similar across categories, 
with no single category standing out as significantly more premium.

**Conclusion:**
Since Running represents Adidas' core market both in volume, 
it is the most relevant segment to analyze for a brand 
looking to compete with Adidas. The following analysis will 
therefore focus exclusively on the Running category.

![descrizione](../images/category_analysis.png)

## Filtering for Running — SQL Query

The following query applies the same cleaning logic as before, 
filtered to the Running category only.

**Key operations:**

**1. Deduplication** — `MIN(price_local)` and `GROUP BY product_id` 
keep only the lowest price per unique product.

**2. Currency conversion to EUR** — A `CASE` statement converts all 
prices to EUR using fixed exchange rates.

**3. Discount handling** — `COALESCE(MAX(discount_pct), 0)` replaces 
NULL discounts with 0.

**4. Filtering** — `WHERE category IN ('Running')` restricts the dataset 
to Running products only.

```sql
SELECT 
    product_id,
    product_name,
    category,
    subcategory,
    gender_segment,
    product_url,
    currency,
    MIN(price_local),
    ROUND(MIN(price_local) * CASE currency
      WHEN 'EUR' THEN 1.0
      WHEN 'USD' THEN 0.92
      WHEN 'GBP' THEN 1.17
      WHEN 'BRL' THEN 0.17
      WHEN 'CNY' THEN 0.13
      WHEN 'INR' THEN 0.011
      WHEN 'JPY' THEN 0.0062
      WHEN 'KRW' THEN 0.00068
      WHEN 'MXN' THEN 0.046
      ELSE 1.0
    END, 2) as price_eur,
    REPLACE(CAST(COALESCE(MAX(discount_pct), 0) AS CHAR), '.', ',') as discount_pct
FROM adidas_global
WHERE subcategory IS NOT NULL AND category IN ('Running')
GROUP BY 
    product_id,
    category,
    subcategory,
    gender_segment,
    currency
```

The query output was exported as a CSV file named `running_data_cleanV3.csv` 
and loaded into Tableau as a new data source for further visual analysis.

## Price Dispersion in the Running Category

The chart above shows the distribution of unique products across price points 
in the Running category.

Two potential clusters appear to emerge visually:
- A first concentration around **65€**
- A second concentration around **138-140€**

![](../images/price_dispersion_running_analysis.png)

This suggests that Adidas structures its Running catalog around two distinct 
price segments: an **entry/mid-range** tier around 65€, and a **premium** tier 
around 138-140€, likely targeting different customer profiles within the 
same category.

## Creating a new Feature

A dedicated table was created from the main dataset, filtering only 
the Running category and applying the same cleaning logic described above.

Since product names appear in multiple languages (English, French, Spanish, 
Portuguese, German, Japanese, Korean, Chinese), a rule-based classification 
was generated with the help of Claude AI, which analyzed all unique product 
names and created a `product_type` feature using multilingual keyword matching.

The feature classifies each product into three categories:
- **Shoes** — running footwear across all languages
- **Apparel** — jackets, shorts, leggings, t-shirts, etc.
- **Accessory** — socks, caps, sunglasses

Products in non-Latin scripts (Japanese, Korean, Chinese) are not classified.

The final table was exported as `running_data_clean.csv` and loaded 
into Tableau as a new data source for visual analysis.

## Price Dispersion by Product Type

Breaking down the price distribution by product type reveals a clear 
and deliberate pricing strategy from Adidas in the Running segment.

The three charts below show the distribution separately for Shoes, 
Apparel, and Accessories.

## General
![](../images/price_dispersion_by_product_type.png)
## Accessory
![](../images/price_dispersion_accessories.png)
## Apparel
![](../images/price_dispersion_apparel.png)
## Shoes
![](../images/price_dispersion_shoes.png)

## Adidas' Pricing Strategy in Running

The data reveals a well-defined two-tier strategy:

**Footwear** is Adidas' core business in Running, with prices ranging 
from 22€ to 500€ and two clear concentrations:

- An **entry-level tier** around **65-76€** — basic running shoes 
designed for everyday training 
- A **premium tier** around **138-150€** — high-performance shoes 
featuring advanced technology targeting experienced runners

## Entry-level shoes
![](../images/entry_level_shoes.png)
## Premium tier shoes
![](../images/premium_shoes.png)

**Apparel** (shorts, t-shirts, jackets, leggings) occupies a much lower 
price range, concentrated between **15€ and 100€**, with most products 
sitting around 60€. This suggests apparel serves more as a complementary 
product line rather than a premium offering.

**Accessories** (socks, caps, sunglasses) are marginal in the catalog 
and cover a very limited price range.

---

## Conclusions — Positioning for a New Brand

Competing directly with Adidas in Running footwear would be extremely 
difficult. Adidas has dominant market presence, strong brand recognition, 
and established technology platforms (Boost, Repetitor, Lightstrike) 
that are hard to replicate for a new entrant.

Two alternative positioning strategies emerge from this analysis:

**Option 1 — Premium Accessories**  
Adidas almost completely ignores accessories in Running. 
A new brand could own this space with high-quality, high-price 
running accessories (advanced GPS watches, premium hydration gear, 
performance eyewear), a segment where Adidas has no real presence.

**Option 2 — Premium Apparel**  
Adidas positions its Running apparel in the low-to-mid price range (15€–100€). 
A new brand could differentiate by offering premium technical running apparel 
at a higher price point (100€–200€), targeting serious runners who are 
willing to pay for quality in clothing as much as in footwear.

Both strategies avoid direct competition with Adidas' strongest asset, 
its footwear, and instead target segments where the brand is either 
absent or underinvested.